In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import torchaudio
from safetensors.torch import save_file
from model_audio_input.model import SpecTTTra
import torch
from model_audio_input.dataset import SonicDataset
from safetensors.torch import load_file
import librosa

In [2]:
def load_model(weights_path: str, num_classes: int = 2) -> nn.Module:
    """Load SpecTTTra with safetensors weights for GASBench-style inference."""
    model = SpecTTTra(
        input_spec_dim=128,
        input_temp_dim=188,
        embed_dim=384,
        f_clip=1,
        t_clip=3,
        num_heads=6,
        num_layers=12,
        num_classes=2,
        expected_samples=96000,
    )
    state_dict = load_file(weights_path)
    model.load_state_dict(state_dict)
    model.eval()
    return model

In [3]:
weight_path = "submit/model.safetensors"
model = load_model(weight_path)
print("Loaded model")

Loaded model


In [22]:
audio_path = "crop_data/crop5/real/real_00358.wav"
waveform, sr = librosa.load(audio_path)   # [C, T]
waveform = torch.as_tensor(waveform, dtype=torch.float32)
if waveform.ndim == 1:
    waveform = waveform.unsqueeze(0)
elif waveform.ndim == 2 and waveform.shape[0] > waveform.shape[1]:
    waveform = waveform.transpose(0, 1)
sample_rate = 16000
expected_samples = 96000

# convert về mono nếu nhiều channel
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0)
else:
    waveform = waveform.squeeze(0)  # [T]

# resample nếu cần
if sr != sample_rate:
    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=sample_rate)
    waveform = resampler(waveform.unsqueeze(0)).squeeze(0)

# Model receives raw waveform and performs mel extraction inside forward().
waveform = waveform.to(torch.float32)
if waveform.numel() < expected_samples:
    pad = expected_samples - waveform.numel()
    waveform = torch.nn.functional.pad(waveform, (0, pad))
elif waveform.numel() > expected_samples:
    waveform = waveform[: expected_samples]


In [23]:
print(waveform.shape)
waveform = waveform.unsqueeze(0)   # [1, 96000]
print(waveform.shape)

torch.Size([96000])
torch.Size([1, 96000])


In [24]:
with torch.no_grad():
    logits = model(waveform)  # [1, 2]
print(logits)
pred = torch.argmax(logits, dim=1)
print(pred)

tensor([[-5.0576,  5.4797]])
tensor([1])
